# Limpeza dos dados - Distribuidora Vale Sul

Lê os CSVs brutos de `data/raw`, aplica as decisões de limpeza e carrega o resultado no banco `data/processed/vale_sul.db`.
Os CSVs brutos não são alterados. Para refazer tudo, rode o notebook desde a primeira célula.

## Leitura dos CSVs

In [11]:
import pandas as pd

clientes = pd.read_csv("../data/raw/clientes.csv", sep=";", decimal=",")
vendedores = pd.read_csv("../data/raw/vendedores.csv", sep=";", decimal=",")
produtos = pd.read_csv("../data/raw/produtos.csv", sep=";", decimal=",")
pedidos = pd.read_csv("../data/raw/pedidos.csv", sep=";", decimal=",")
itens = pd.read_csv("../data/raw/itens_pedido.csv", sep=";", decimal=",")
titulos = pd.read_csv("../data/raw/titulos_receber.csv", sep=";", decimal=",")
pagamentos = pd.read_csv("../data/raw/pagamentos.csv", sep=";", decimal=",")

## Tarefas da limpeza

### 1. Clientes
1. CNPJ: deixar só os números (tirar pontos, barra e traço). Vem antes dos duplicados, senão parte deles fica escondida.
2. Tirar espaços sobrando no começo e no fim dos textos.
3. Cidade "SP": considerar São Paulo.
4. Cidade e segmento com grafias diferentes: unificar para a grafia que mais aparece na base.
5. UF vazia (25): preencher com "SP".
6. Segmento vazio (14) e vendedor_id vazio (9): manter vazio (nulo).
7. Clientes duplicados (28): juntar os cadastros com o mesmo CNPJ, que fica único no banco. Trocar o `cliente_id` antigo pelo novo em `pedidos` e em `titulos`.

### 2. Pedidos
8. 3 pedidos com ano 2052 (133467, 143683, 156644): corrigir para 2025. Vem antes do custo dos produtos, que usa a data do pedido.

### 3. Produtos
9. 6 produtos sem categoria: descobrir a categoria pela descrição.
10. Custo maior que o preço no cadastro (37, 40, 134, 216): corrigir com o custo da última venda (`itens.custo_unitario`, usando a data de `pedidos`).
11. Produtos 901, 902 e 903 (não existem no cadastro): criar como "produto desconhecido", com categoria "Desconhecido".
12. `ativo`: converter S/N para 1/0 (o banco só aceita 0 e 1).

### 4. Itens do pedido
13. 9 itens com quantidade 100 vezes maior: dividir a quantidade por 100.

### 5. Pagamentos
14. 1.195 datas no formato dd/mm/aaaa: converter cada formato separado e juntar com `fillna`. Vem antes dos duplicados, porque 5 deles só ficam iguais depois da conversão.
15. 45 pagamentos duplicados: remover.

### 6. Títulos a receber
16. Status recalculado a partir da soma dos pagamentos (30 estavam errados). Depende dos pagamentos já sem duplicados.
17. 15 títulos de pedidos cancelados: status `CANCELADO`. Vem depois do recálculo, senão o recálculo apaga essa marcação.

### 7. Carga no banco
18. Carregar na ordem das ligações entre tabelas: vendedores → clientes → produtos → pedidos → itens → títulos → pagamentos.

## 1. Clientes

### Tarefa 1 - CNPJ só com números

In [12]:
clientes["cnpj"] = clientes["cnpj"].str.replace(r"\D", "", regex=True)

In [13]:
# Conferência: CNPJs que ainda têm pontuação (esperado: 0)
clientes["cnpj"].str.contains(r"[./-]").sum()

np.int64(0)

### Tarefa 2 - Tirar espaços sobrando no começo e no fim dos textos

In [14]:
for coluna in ["razao_social", "nome_fantasia", "cnpj", "segmento", "cidade", "uf"]:
    clientes[coluna] = clientes[coluna].str.strip()

### Tarefas 3 e 4 - Padronizar cidade e segmento

Cada grafia errada aponta para a grafia que mais aparece na base. "SP" na cidade é considerado São Paulo.

In [15]:
padrao_cidade = {
    "Sao Paulo": "São Paulo",
    "S. Paulo": "São Paulo",
    "SP": "São Paulo",
    "SAO PAULO": "São Paulo",
    "GUARULHOS": "Guarulhos",
    "MOGI DAS CRUZES": "Mogi das Cruzes",
    "STO ANDRE": "Santo André",
    "Santo Andre": "Santo André",
    "S. Bernardo do Campo": "São Bernardo do Campo",
    "SBC": "São Bernardo do Campo",
    "Sao Bernardo do Campo": "São Bernardo do Campo",
    "Mogi": "Mogi das Cruzes"
}
clientes["cidade"] = clientes["cidade"].replace(padrao_cidade)

In [16]:
# Conferência: devem sobrar só as 12 cidades
clientes["cidade"].value_counts()

cidade
São Paulo                516
Guarulhos                125
São Bernardo do Campo     94
Osasco                    82
Santo André               75
Mogi das Cruzes           64
Barueri                   60
Suzano                    48
Diadema                   47
Mauá                      43
Itaquaquecetuba           38
Carapicuíba               36
Name: count, dtype: int64

In [17]:
padrao_segmento = {
    "Bar/Restaurante": "Bar e Restaurante",
    "MERCADO": "Mercado",
    "Conveniencia": "Conveniência",
    "padaria": "Padaria"
}
clientes["segmento"] = clientes["segmento"].replace(padrao_segmento)

In [18]:
# Conferência: devem sobrar só os 4 segmentos, mais os 14 vazios
clientes["segmento"].value_counts(dropna=False)

segmento
Bar e Restaurante    447
Mercado              364
Padaria              218
Conveniência         185
NaN                   14
Name: count, dtype: int64

### Tarefa 5 - UF vazia: preencher com "SP"

In [19]:
clientes["uf"] = clientes["uf"].fillna("SP")

In [20]:
clientes["uf"].isna().sum()

np.int64(0)